    
Goal: Test specificity

In [ ]:
# libraries 

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact, ttest_ind # for r-to-z and t-test
from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import multipletests
import numpy as np
from scipy.stats import percentileofscore
    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.percentileofscore.html

In [ ]:
# lists

targets = ["P03_anger", "P03_joy", "P03_shame", "P12_neutral", "P12_sadness", "P15_anger", "P15_sadness",
           "P17_neutral", "P17_pride", "P22_anger", "P22_pride", "P22_shame", "P27_sadness", "P29_joy",
           "P29_neutral", "P29_pride", "P34_joy", "P34_neutral", "P34_sadness", "P37_shame", "P38_anger",
           "P38_neutral", "P38_shame", "P42_anger", "P42_joy", "P42_pride", "P44_joy", "P44_pride",
           "P44_sadness", "P45_shame"]

perceivers = ["S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08", "S11", "S15", "S18", "S20", "S23", 
                "S24", "S28", "S34", "S37", "S42", "S44", "S45", "S47", "S49", "S51", "S54", "S55", "S57", "S64", "S65"]

DISCARD_PERCEIVERS = ['S01', 'S18'] # removing perceivers with >5% removed ECG data
perceivers = [perceiver for perceiver in perceivers if perceiver not in DISCARD_PERCEIVERS]

# data from previous script
HR_data = pd.read_csv("outputs/RMSSD_synchrony_scores.csv")
HR_data = HR_data[~HR_data["perceiver"].isin(["S01", "S18"])] # removing S01 and S18 from analyses (just in case it's still there)

HRV_data = pd.read_csv("outputs/BPM_synchrony_scores.csv")
HRV_data = HRV_data[~HRV_data["perceiver"].isin(["S01", "S18"])] # removing S01 and S18 from analyses (just in case it's still there)

r-to-z transformation

In [ ]:

def r_to_z(r_value):
    return np.log((1 + r_value) / (1 - r_value)) / 2

HR_data['z_corr'] = r_to_z(HR_data['spearman_r'])

# plot r-values and z-values
plt.figure(figsize=(7, 5))
plt.hist(HR_data['spearman_r'], bins=30, color='blue', label='r') # r-values
plt.hist(HR_data['z_corr'], bins=30, color='red', label='z', alpha=.3) # z-values
plt.xlabel('value')
plt.ylabel('count')
plt.title('raw r-values vs fisher z-values')
plt.legend()
plt.show()

## same for rmssd

HRV_data['z_corr'] = r_to_z(HRV_data['spearman_r'])

# plot r-values and z-values
plt.figure(figsize=(7, 5))
plt.hist(HRV_data['spearman_r'], bins=30, color='blue', label='r') # r-values
plt.hist(HRV_data['z_corr'], bins=30, color='red', label='z', alpha=.3) # z-values
plt.xlabel('value')
plt.ylabel('count')
plt.title('raw r-values vs fisher z-values')
plt.legend()
plt.show()



Graph to show the distribution of matching and nonmatching correlations. 

matching: correlation values betweeen perceivers and the target they actually watched\
nonmatching: surrogate correlations between perceivers and targets they did not watch\
I also look at surrogate correlations with targets of the same emotion category as the one the perceiver actually watched. 

In [ ]:
def match_category(row):
    if row['match_type'] == 'match':
        return 'match' # matching correlation
    elif row['emotion_type'] == 'same_emotion' and row['match_type'] == 'nonmatch':
        return 'nonmatch_same' # nonmatching (and not same emotion as the actual target)
    elif row['emotion_type'] == 'diff_emotion':
        return 'nonmatch_diff' # nonmatching (and same emotion as the actual target)
    
# put this information in a new column
HR_data['match_category'] = HR_data.apply(match_category, axis=1) # axis 1 = apply to rows

# make the graph
nrow = 6 
ncol = 5 
fig, axs = plt.subplots(nrow, ncol, figsize=(13, 15))
axs = axs.flatten()  # numbered list of plots; easier for looping

for i, ax in enumerate(axs):
    target = targets[i]
    data = HR_data[HR_data['evaluation_of'] == target] # select data for this target only

    ax.hist(data[data['match_category'] == 'nonmatch_diff']['z_corr'], # diff color for match, nonmatch and same emotion
            bins=30, alpha=0.4, color='navy', label='nonmatch_diff', edgecolor='none')

    ax.hist(data[data['match_category'] == 'nonmatch_same']['z_corr'],
            bins=30, alpha=0.5, color='cyan', label='nonmatch_same', edgecolor='none')

    ax.hist(data[data['match_category'] == 'match']['z_corr'],
            bins=30, alpha=1.0, color='red', label='match', edgecolor='none')

    ax.set_title(target)
    ax.axvline(0, color='gray', linestyle='--')
    ax.set_ylim(0,40)
    ax.set_xlim(-3,3)

fig.suptitle('Matching vs. Nonmatching Distribution (HR synchrony)', fontsize=16) # figure
fig.legend(['nonmatch_diff', 'nonmatch_same', 'match'], loc='upper right')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


### same for rmssd

HRV_data['match_category'] = HRV_data.apply(match_category, axis=1)

# make the graph
nrow = 6 
ncol = 5 
fig, axs = plt.subplots(nrow, ncol, figsize=(13, 15))
axs = axs.flatten()  # numbered list of plots; easier for looping

for i, ax in enumerate(axs):
    target = targets[i]
    data = HRV_data[HRV_data['evaluation_of'] == target]

    ax.hist(data[data['match_category'] == 'nonmatch_diff']['z_corr'],
            bins=30, alpha=0.4, color='navy', label='nonmatch_diff', edgecolor='none')

    ax.hist(data[data['match_category'] == 'nonmatch_same']['z_corr'],
            bins=30, alpha=0.5, color='cyan', label='nonmatch_same', edgecolor='none')

    ax.hist(data[data['match_category'] == 'match']['z_corr'],
            bins=30, alpha=1.0, color='red', label='match', edgecolor='none')

    ax.set_title(target)
    ax.axvline(0, color='gray', linestyle='--')
    ax.set_ylim(0,40)
    ax.set_xlim(-3,3)

fig.suptitle('Matching vs. Nonmatching Distribution (HRV synchrony)', fontsize=16)
fig.legend(['nonmatch_diff', 'nonmatch_same', 'match'], loc='upper right')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

We want to test for target-specificity. Step #1: Calculate percentile value of each target-perceiver pair. And add this value in a new column in the EA scores dataset.\
Also check percentile distribution within target. 

In [ ]:


full_distribution_HR = HR_data['z_corr']
percentile_in_full_distribution_HR = [] # percentile of a value in the full distribution
percentile_in_target_HR = [] # percentile of a value relative to just the corresponding target

# go through each row and calculate percentile 
for idx, row in HR_data.iterrows():
    z = row['z_corr']
    target = row['evaluation_of']
    
    # percentile in full data
    percentile_HR = percentileofscore(full_distribution_HR, z, kind='mean')
    percentile_in_full_distribution_HR.append(percentile_HR)

    # percentile in target data (all data just for the corresponding target)
    target_dist = HR_data[HR_data['evaluation_of'] == target]['z_corr']
    percentile_HR_target = percentileofscore(target_dist, z, kind='mean')
    percentile_in_target_HR.append(percentile_HR_target)
    
# add to current dataframe
HR_data['percentile_in_full_dist_HR'] = percentile_in_full_distribution_HR
HR_data['percentile_in_target_dist_HR'] = percentile_in_target_HR

# re-export with added column
HR_data.to_csv("outputs/BPM_synchrony_scores.csv", index=False)

############################ same for HRV scores
full_distribution_HRV = HRV_data['z_corr']
percentile_in_full_distribution_HRV = []
percentile_in_target_HRV = [] 

for idx, row in HRV_data.iterrows():
    score = row['z_corr']
    target = row['evaluation_of']

    # percentile in full dataset
    percentile_HRV = percentileofscore(full_distribution_HRV, score, kind='mean')
    percentile_in_full_distribution_HRV.append(percentile_HRV)

    # percentile in target data
    target_dist = HRV_data[HRV_data['evaluation_of'] == target]['z_corr']
    percentile_HRV_target = percentileofscore(target_dist, score, kind='mean')
    percentile_in_target_HRV.append(percentile_HRV_target)
    
# add to current dataframe
HRV_data['percentile_in_full_dist_HRV'] = percentile_in_full_distribution_HRV
HRV_data['percentile_in_target_dist_HRV'] = percentile_in_target_HRV

# re-export with added column
HRV_data.to_csv("outputs/RMSSD_synchrony_scores.csv", index=False)

One-sample t-test for each measure, to test if matching scores are significantly higher than the null distribution

In [ ]:

# correlation EA scores
full_distribution_HR = HR_data['percentile_in_full_dist_HR']
match_distribution_HR = HR_data[HR_data['match_type'] == 'match']['percentile_in_full_dist_HR'].dropna()
match_avg_corr = (
    HR_data[HR_data['match_type'] == 'match']
    .dropna(subset=['percentile_in_full_dist_HR'])
    .groupby('perceiver')['percentile_in_full_dist_HR']
    .mean())
t_value, p_value = ttest_1samp(match_avg_corr, popmean=full_distribution_HR.mean())

# print results
dof = len(match_avg_corr) - 1 # degrees of freedom
print(f"Matching: M = {match_avg_corr.mean():.2f}, SD = {match_avg_corr.std():.2f}")
print(f"Full distribution: M = {full_distribution_HR.mean():.2f}, SD = {full_distribution_HR.std():.2f}")
print(f"One-sample t-test: t({dof}) = {t_value:.2f}, p = {p_value:.4f}")

# make it into a graph
groups = ['Matching Scores', 'Full Distribution']
data = [match_avg_corr, full_distribution_HR]

plt.figure(figsize=(5.5, 6))

for i, values in enumerate(data): # loop through values of each group
    
    # 50% box (from percentile 25 to 75)
    plt.fill_between([i - 0.2, i + 0.2], # add thickness
                     np.percentile(values, 25), # starts at percentile 25
                     np.percentile(values, 75), # ends at percentile 75
                     color='lightgray', alpha=1.0, edgecolor='black', linewidth=1)

    # horizontal mean line
    plt.plot([i - 0.2, i + 0.2], 
             [values.mean(), values.mean()], 
             color='black', linewidth=2)

    # sd vertical lines
    plt.plot([i, i], [np.percentile(values, 75), # start outside the box
                      values.mean() + values.std()], color='black', linewidth=1.2)
    plt.plot([i, i], [np.percentile(values, 25), # start outside the box
                      values.mean() - values.std()], color='black', linewidth=1.2)

    # horizontal line at +/- 1 SD
    plt.plot([i - 0.1, i + 0.1], # not as wide as the box
             [values.mean() + values.std(), 
              values.mean() + values.std()], 
             color='black', linewidth=1)
    plt.plot([i - 0.1, i + 0.1], 
             [values.mean() - values.std(), 
              values.mean() - values.std()], 
             color='black', linewidth=1)
    
ax = plt.gca()  # get axes
ax.grid(False) # remove background grid

plt.xticks([0, 1], groups, fontsize=12) # label each group
plt.yticks(np.arange(0, 110, 10), fontsize=10) # ticks at each jump of 10 (110 so we can see '100')
plt.ylabel('Percentile', fontsize=12)
plt.axhline(50, color='gray', linestyle=':', linewidth=2)  # dashed line at 50
plt.title('Percentile Distribution of Matching scores (HR synchrony) vs. Full Distribution', fontsize=13, pad=15)
plt.ylim(0, 100)
plt.show()

############################################## same with HRV

full_distribution_HRV = HRV_data['percentile_in_full_dist_HRV']
match_distribution_HRV = HRV_data[HRV_data['match_type'] == 'match']['percentile_in_full_dist_HRV'].dropna()
match_avg_corr = (
    HRV_data[HRV_data['match_type'] == 'match']
    .dropna(subset=['percentile_in_full_dist_HRV'])
    .groupby('perceiver')['percentile_in_full_dist_HRV']
    .mean())
t_value, p_value = ttest_1samp(match_avg_corr, popmean=full_distribution_HRV.mean())

# print results
dof = len(match_avg_corr) - 1 # degrees of freedom
print(f"Matching: M = {match_avg_corr.mean():.2f}, SD = {match_avg_corr.std():.2f}")
print(f"Full distribution: M = {full_distribution_HRV.mean():.2f}, SD = {full_distribution_HRV.std():.2f}")
print(f"One-sample t-test: t({dof}) = {t_value:.2f}, p = {p_value:.4f}")

# make it into a graph
groups = ['Matching Scores', 'Full Distribution']
data = [match_avg_corr, full_distribution_HRV]

plt.figure(figsize=(5.5, 6))

for i, values in enumerate(data): # loop through values of each group
    
    # 50% box (from percentile 25 to 75)
    plt.fill_between([i - 0.2, i + 0.2], # add thickness
                     np.percentile(values, 25), # starts at percentile 25
                     np.percentile(values, 75), # ends at percentile 75
                     color='lightgray', alpha=1.0, edgecolor='black', linewidth=1)

    # horizontal mean line
    plt.plot([i - 0.2, i + 0.2], 
             [values.mean(), values.mean()], 
             color='black', linewidth=2)

    # sd vertical lines
    plt.plot([i, i], [np.percentile(values, 75), # start outside the box
                      values.mean() + values.std()], color='black', linewidth=1.2)
    plt.plot([i, i], [np.percentile(values, 25), # start outside the box
                      values.mean() - values.std()], color='black', linewidth=1.2)

    # horizontal line at +/- 1 SD
    plt.plot([i - 0.1, i + 0.1], # not as wide as the box
             [values.mean() + values.std(), 
              values.mean() + values.std()], 
             color='black', linewidth=1)
    plt.plot([i - 0.1, i + 0.1], 
             [values.mean() - values.std(), 
              values.mean() - values.std()], 
             color='black', linewidth=1)

ax = plt.gca()  # get axes
ax.grid(False) # remove background grid

plt.xticks([0, 1], groups, fontsize=12) # label each group
plt.yticks(np.arange(0, 110, 10), fontsize=10) # ticks at each jump of 10 (110 so we can see '100')
plt.ylabel('Percentile', fontsize=12)
plt.axhline(50, color='gray', linestyle=':', linewidth=2)  # dashed line at 50
plt.title('Percentile Distribution of Matching scores (HRV synchrony) vs. Full Distribution', fontsize=13, pad = 15)
plt.ylim(0, 100)
plt.show()

Specificity per target (exploratory)

In [ ]:
# testing the specificity for each video individually (real pairs vs. surrogate pairs)

target_specificity_results_HR = []

for target in targets:
    
    full_distribution = HR_data[HR_data['evaluation_of'] == target]["percentile_in_target_dist_HR"].dropna()
    match_distribution = HR_data[(HR_data['evaluation_of'] == target) & (HR_data['match_type'] == 'match')]["percentile_in_target_dist_HR"].dropna()

    t_value, p_value = ttest_1samp(match_distribution, popmean=full_distribution.mean())
    
    target_specificity_results_HR.append({
        "target": target,
        "mean_matching": match_distribution.mean(),
        "sd_matching": match_distribution.std(),
        "mean_full_dist": full_distribution.mean(),
        "sd_full_dist": full_distribution.std(),
        "n": len(match_distribution),
        "t": t_value,
        "p": p_value})

target_spec_HR = pd.DataFrame(target_specificity_results_HR)

reject, p_corr, _, _ = multipletests(target_spec_HR['p'], method='fdr_bh')
target_spec_HR['p_corrected'] = p_corr

print(target_spec_HR)

###################### same thing for HRV

target_specificity_results_HRV = []

for target in targets:
    
    full_distribution = HRV_data[HRV_data['evaluation_of'] == target]["percentile_in_target_dist_HRV"].dropna()
    match_distribution = HRV_data[(HRV_data['evaluation_of'] == target) & (HRV_data['match_type'] == 'match')]["percentile_in_target_dist_HRV"].dropna()

    t_value, p_value = ttest_1samp(match_distribution, popmean=full_distribution.mean())
    
    target_specificity_results_HRV.append({
        "target": target,
        "mean_matching": match_distribution.mean(),
        "sd_matching": match_distribution.std(),
        "mean_full_dist": full_distribution.mean(),
        "sd_full_dist": full_distribution.std(),
        "n": len(match_distribution),
        "t": t_value,
        "p": p_value})

target_spec_HRV = pd.DataFrame(target_specificity_results_HRV)

reject, p_corr, _, _ = multipletests(target_spec_HRV['p'], method='fdr_bh')
target_spec_HRV['p_corrected'] = p_corr

print(target_spec_HRV)

GRAPHS per target

In [ ]:

# only keep matching scores
match_data = HR_data[HR_data['match_type'] == 'match']

fig, ax = plt.subplots(figsize=(15, 8))

for i, target in enumerate(targets):
    
    values = match_data[match_data['evaluation_of'] == target]['percentile_in_target_dist_HR'].dropna()

    # different color depending on p value
    p = target_spec_HR[target_spec_HR['target'] == target]['p_corrected'].values[0]
    color = 'gray' if p < 0.05 else 'red'
    
    # 50% box
    ax.fill_between([i - 0.2, i + 0.2], np.percentile(values, 25), np.percentile(values, 75), color=color, edgecolor='black', linewidth=0.8)
    
    # mean horizontal line
    ax.plot([i - 0.2, i + 0.2], [np.mean(values), np.mean(values)], color='black', linewidth=2)
    
    # SD if outside box (some boxes are longer than SD from mean)
    if np.mean(values) + np.std(values) > np.percentile(values, 75):
        ax.plot([i, i], [np.percentile(values, 75), np.mean(values) + np.std(values)], color='black')
        ax.plot([i - 0.1, i + 0.1], [np.mean(values) + np.std(values), np.mean(values) + np.std(values)], color='black')
    if np.mean(values) - np.std(values) < np.percentile(values, 25):
        ax.plot([i, i], [np.mean(values) - np.std(values), np.percentile(values, 25)], color='black')
        ax.plot([i - 0.1, i + 0.1], [np.mean(values) - np.std(values), np.mean(values) - np.std(values)], color='black')

ax.grid(False) # remove background grid

ax.set_xticks(np.arange(len(targets)))
ax.set_xticklabels(targets, rotation=90)
ax.set_ylim(0, 110) # 110 so we can see '100'
ax.axhline(50, linestyle='--', color='gray')
ax.set_ylabel("Percentile")
ax.set_title("HR synchrony – Match Percentiles per Target")

ax.bar(0, 0, color='gray', label='Significant (p<.05)')
ax.bar(0, 0, color='red', label='Not significant (p>.05)')
ax.legend(loc='upper right')
plt.show()


###### same thing for HRV

# only keep matching scores
match_data = HRV_data[HRV_data['match_type'] == 'match']

fig, ax = plt.subplots(figsize=(15, 8))

for i, target in enumerate(targets):
    
    values = match_data[match_data['evaluation_of'] == target]['percentile_in_target_dist_HRV'].dropna()

    # different color depending on p value
    p = target_spec_HRV[target_spec_HRV['target'] == target]['p_corrected'].values[0]
    color = 'gray' if p < 0.05 else 'red'
    
    # 50% box
    ax.fill_between([i - 0.2, i + 0.2], np.percentile(values, 25), np.percentile(values, 75), color=color, edgecolor='black', linewidth=0.8)
    
    # mean horizontal line
    ax.plot([i - 0.2, i + 0.2], [np.mean(values), np.mean(values)], color='black', linewidth=2)
    
    # SD if outside box (some boxes are longer than SD from mean)
    if np.mean(values) + np.std(values) > np.percentile(values, 75):
        ax.plot([i, i], [np.percentile(values, 75), np.mean(values) + np.std(values)], color='black')
        ax.plot([i - 0.1, i + 0.1], [np.mean(values) + np.std(values), np.mean(values) + np.std(values)], color='black')
    if np.mean(values) - np.std(values) < np.percentile(values, 25):
        ax.plot([i, i], [np.mean(values) - np.std(values), np.percentile(values, 25)], color='black')
        ax.plot([i - 0.1, i + 0.1], [np.mean(values) - np.std(values), np.mean(values) - np.std(values)], color='black')


ax.grid(False) # remove background grid

ax.set_xticks(np.arange(len(targets)))
ax.set_xticklabels(targets, rotation=90)
ax.set_ylim(0, 110) # 110 so we can see '100'
ax.axhline(50, linestyle='--', color='gray')
ax.set_ylabel("Percentile")
ax.set_title("HRV synchrony – Match Percentiles per Target")

ax.bar(0, 0, color='gray', label='Significant (p<.05)')
ax.bar(0, 0, color='red', label='Not significant (p>.05)')
ax.legend(loc='upper right')
plt.show()